In [6]:
import pandas as pd

peaks_atac = pd.read_csv("/home/elicanter/datasets/paul2015data/atac_matrix.binary.qc_filtered.peaks.txt", sep="\t")

In [3]:
peaks_atac2 = pd.read_csv("/home/elicanter/datasets/hematopoiesis/GSE129785_scATAC-Hematopoiesis-CD34.peaks.txt.gz", sep="\t")
peaks_atac2

,Feature
0,chr1_10238_10738
1,chr1_115476_115976
2,chr1_236856_237356
3,chr1_237511_238011
4,chr1_240801_241301
...,...
571384,chrX_155249323_155249823
571385,chrX_155253875_155254375
571386,chrX_155254735_155255235
571387,chrX_155259327_155259827


In [7]:
peaks_atac


,Feature
0,chr1_3002478_3002968
1,chr1_3084739_3085712
2,chr1_3103576_3104022
3,chr1_3106871_3107210
4,chr1_3108932_3109158
...,...
436201,chrY_621772_623366
436202,chrY_629871_630911
436203,chrY_631222_631480
436204,chrY_795887_796426


In [5]:
print(peaks_atac2.head())
print(peaks_atac2.columns)

              Feature
0    chr1_10238_10738
1  chr1_115476_115976
2  chr1_236856_237356
3  chr1_237511_238011
4  chr1_240801_241301
Index(['Feature'], dtype='object')


In [9]:
import pandas as pd
! wget https://raw.githubusercontent.com/morris-lab/CellOracle/master/docs/demo_data/TF_data_in_Paul15.csv
df = pd.read_csv("TF_data_in_Paul15.csv")

print(df["TF"].nunique())
print(df["TF"].unique())

--2026-07-10 15:20:05--  https://raw.githubusercontent.com/morris-lab/CellOracle/master/docs/demo_data/TF_data_in_Paul15.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1771 (1.7K) [text/plain]
Saving to: ‘TF_data_in_Paul15.csv’

TF_data_in_Paul15.c 100%[===================>]   1.73K  --.-KB/s    in 0s      

2026-07-10 15:20:05 (33.7 MB/s) - ‘TF_data_in_Paul15.csv’ saved [1771/1771]



4
['Cebpa' 'Irf8' 'Klf1' 'Spi1']


In [11]:
import anndata as ad

celloracle_tfs = set(df["TF"].str.upper())
rna_adata = ad.read_h5ad("/home/elicanter/datasets/paul2015data/Paul_etal_15.h5ad")
your_rna = {g.upper() for g in rna_adata.var_names}

print(len(celloracle_tfs & your_rna))

3


In [19]:
rna_adata

AnnData object with n_obs × n_vars = 2671 × 1999
    obs: 'paul15_clusters', 'n_counts_all', 'n_counts', 'louvain', 'cell_type'
    var: 'n_counts', 'mean', 'std'
    uns: 'cell_type_colors', 'diffmap_evals', 'draw_graph', 'iroot', 'log1p', 'louvain', 'louvain_colors', 'louvain_sizes', 'neighbors', 'paga', 'paul15_clusters_colors', 'pca'
    obsm: 'X_diffmap', 'X_draw_graph_fa', 'X_draw_graph_fr', 'X_pca'
    varm: 'PCs'
    layers: 'raw_count', 'raw_count_int'
    obsp: 'connectivities', 'distances'

In [12]:
celloracle_tfs = set(df["TF"].astype(str))
rna_genes = set(rna_adata.var_names.astype(str))

print("TFs:", len(celloracle_tfs))
print("RNA genes:", len(rna_genes))
print("Overlap:", len(celloracle_tfs & rna_genes))

print("Examples missing:")
print(list(celloracle_tfs - rna_genes)[:50])

TFs: 4
RNA genes: 1999
Overlap: 3
Examples missing:
['Spi1']


In [13]:
motif_tf_names =open("/home/elicanter/datasets/paul2015data/tf_names.txt").read().strip().split()

In [14]:
# after cleaning motif names and matching case-insensitively
rna_upper_to_original = {
    g.upper(): g for g in rna_adata.var_names
}

matched = [
    rna_upper_to_original[tf.upper()]
    for tf in motif_tf_names
    if tf.upper() in rna_upper_to_original
]

print(len(set(matched)))
print(sorted(set(matched))[:100])

51
['Batf3', 'Cebpa', 'Cebpe', 'Cux1', 'E2f4', 'Egr1', 'Elf1', 'Elk3', 'Ets1', 'Etv6', 'Fli1', 'Foxp1', 'Gata1', 'Gata2', 'Gfi1', 'Hlf', 'Hnf4a', 'Hoxa5', 'Hoxa7', 'Irf1', 'Irf2', 'Irf7', 'Irf8', 'Irf9', 'Klf2', 'Klf4', 'Klf6', 'Mef2c', 'Meis1', 'Myb', 'Myc', 'Mycn', 'Nfatc2', 'Nfatc3', 'Nfe2', 'Nfia', 'Nfic', 'Nfkb1', 'Nfya', 'Nr3c1', 'Pbx1', 'Rara', 'Rel', 'Rreb1', 'Runx2', 'Sox13', 'Sox4', 'Stat1', 'Stat3', 'Tcf7', 'Zbtb7a']


In [15]:
for tf in ["Spi1", "Klf1", "Tal1", "Ikzf1", "Ikzf2", "Bcl11a"]:
    print(tf, tf.upper() in {x.upper() for x in rna_adata.var_names})

Spi1 False
Klf1 True
Tal1 False
Ikzf1 False
Ikzf2 True
Bcl11a True


In [16]:
matched = set([x.upper() for x in A_tfs])
rna = {x.upper() for x in rna_adata.var_names}

for tf in ["Klf1", "Ikzf2", "Bcl11a"]:
    print(tf, tf.upper() in matched)

NameError: name 'A_tfs' is not defined

In [18]:
for tf in ["KLF1", "IKZF2", "BCL11A"]:
    print(tf, tf in {x.upper() for x in motif_tf_names})

KLF1 False
IKZF2 False
BCL11A False


In [1]:
import anndata as ad
rna_adata = ad.read_h5ad("/home/elicanter/datasets/paul2015data/Paul_etal_15_2.h5ad")
rna_adata

AnnData object with n_obs × n_vars = 2671 × 1999
    obs: 'paul15_clusters', 'n_counts_all', 'n_counts', 'louvain', 'cell_type', 'dpt_pseudotime'
    var: 'n_counts', 'mean', 'std'
    uns: 'cell_type_colors', 'diffmap_evals', 'draw_graph', 'iroot', 'log1p', 'louvain', 'louvain_colors', 'louvain_sizes', 'neighbors', 'paga', 'paul15_clusters_colors', 'pca'
    obsm: 'X_diffmap', 'X_draw_graph_fa', 'X_draw_graph_fr', 'X_pca'
    varm: 'PCs'
    layers: 'raw_count', 'raw_count_int'
    obsp: 'connectivities', 'distances'